# Goal adherence tracking

Track the 14-day moving-average completion rate for all occurrences of one or more recurring goals, with one trace per goal on the same plot.

Each occurrence of a goal has a `target_date` and is resolved as either **completed** (`completed_on` set) or **abandoned** (`abandoned_on` set). The completion rate over a trailing 14-day window is the fraction of resolved occurrences in that window that were completed.

In [ ]:
import dijido
import pandas as pd
import plotly.graph_objects as go

dijido.login()

In [ ]:
# Goals to analyze, mapping each dijido goal name -> the label shown in the
# plot legend. One trace per goal. Use the same string for both if you don't
# want to rename it in the legend.
GOAL_NAMES = {
    "Eat 1 can of salmon": "Eat High Vit D Food",
    "Get 15 min of sun": "Get 15 min of sun",
    "Eat 1 serving of red meat": "Eat Red Meat",
    "Eat at least 1 egg": "Eat 1 Egg",
    "Eat 1 serving vitamin c (e.g. kiwi, OJ)": "Eat High Vit C Food",
}

# Trailing window length (in days) for the moving-average completion rate.
# Larger windows produce a smoother, less jagged curve.
WINDOW_DAYS = 28

In [ ]:
def get_completion_rate(goal_name, window_days=WINDOW_DAYS):
    """Trailing `window_days` moving-average completion rate for a goal.

    Queries dijido for every occurrence of `goal_name`, keeps the resolved ones
    (completed or abandoned), and returns a tuple of:
      - a dataframe with one row per occurrence: ["date", "completion_rate"]
      - the overall adherence (completed / resolved) across all occurrences.
    """
    goals = dijido.get_goals_by_name(goal_name)
    if not goals:
        raise ValueError(f"No goals found named {goal_name!r}")

    # One row per *resolved* occurrence. A still-pending occurrence (e.g.
    # today's) has neither timestamp and is skipped.
    rows = []
    for goal in goals:
        completed_on = goal.get("completed_on")
        abandoned_on = goal.get("abandoned_on")

        if not completed_on and not abandoned_on:
            continue

        rows.append({
            "date": goal["target_date"],
            "completed": completed_on is not None,
        })

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    df["completed"] = df["completed"].astype(int)
    df = df.sort_values("date").set_index("date")

    print(
        f"{goal_name!r}: {len(df)} resolved occurrences "
        f"({int(df['completed'].sum())} completed, "
        f"{int((1 - df['completed']).sum())} abandoned)"
    )

    completion_rate = df["completed"].rolling(f"{window_days}D").mean()
    df_rate = completion_rate.reset_index()
    df_rate.columns = ["date", "completion_rate"]

    # Overall adherence across all resolved occurrences (completed / resolved).
    overall_mean = df["completed"].mean()
    return df_rate, overall_mean

In [ ]:
# Compute the moving-average completion rate and overall adherence per goal.
rates_by_goal = {}
mean_by_goal = {}
for name in GOAL_NAMES:
    df_rate, overall_mean = get_completion_rate(name)
    rates_by_goal[name] = df_rate
    mean_by_goal[name] = overall_mean

In [ ]:
# Qualitative color palette cycled across goals (one color per trace).
COLORS = [
    "#2563eb",  # blue
    "#059669",  # green
    "#dc2626",  # red
    "#d97706",  # amber
    "#7c3aed",  # violet
    "#0891b2",  # cyan
    "#db2777",  # pink
    "#65a30d",  # lime
]

In [ ]:
label_font = dict(size=14, family="Arial Black, Arial, sans-serif", color="#111827")

# Order traces by overall adherence (highest first) so the legend is ranked.
ordered_goals = sorted(rates_by_goal, key=lambda n: mean_by_goal[n], reverse=True)

fig = go.Figure()
for i, goal_name in enumerate(ordered_goals):
    df_rate = rates_by_goal[goal_name]
    color = COLORS[i % len(COLORS)]
    label = f"{GOAL_NAMES[goal_name]} ({mean_by_goal[goal_name]:.0%})"
    fig.add_trace(
        go.Scatter(
            x=df_rate["date"],
            y=df_rate["completion_rate"],
            mode="lines",
            name=label,
            line=dict(color=color, width=2),
            marker=dict(size=7, color=color, line=dict(width=1, color="white")),
        )
    )

fig.update_layout(
    title=dict(
        text=f"{WINDOW_DAYS}-Day Moving Average Adherence Rate",
        font=dict(size=14),
    ),
    xaxis_title=dict(text="Date", font=label_font),
    yaxis_title=dict(text="Adherence rate", font=label_font),
    width=1280,
    height=480,
    margin=dict(l=60, r=30, t=50, b=45),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color="#111827", size=11),
    legend=dict(bgcolor="white", borderwidth=0),
    xaxis=dict(
        showgrid=True,
        gridcolor="#f3f4f6",
        linecolor="#e5e7eb",
        tickfont=label_font,
    ),
    yaxis=dict(
        range=[0, 1],
        tickformat=".0%",
        showgrid=True,
        gridcolor="#f3f4f6",
        linecolor="#e5e7eb",
        tickfont=dict(size=10),
    ),
)
fig.show()